In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import sys
import json
from shapely.geometry import shape
from hotelling.spatial.admin import join_lor_names

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from hotelling.spatial.boundaries import load_boundary

PATH_RAW = REPO_ROOT / Path('data/raw')
PATH_PROCESSED = REPO_ROOT / Path('data/processed')

# Midpoint table (center coordinates)
zensus = gpd.read_parquet(PATH_RAW / 'zensus2022_grid.parquet')
zensus_filtered = gpd.read_parquet(PATH_RAW / 'zensus2022_grid_filtered.parquet')
lor = gpd.read_parquet(PATH_PROCESSED / 'lor.parquet')

# CRITICAL FIX: berlin.geojson has EPSG:3035 coordinates but geopandas auto-detects as EPSG:4326
# We must force the correct CRS instead of transforming from the wrong one
with open(PATH_RAW / 'city_boundary_Berlin.geojson', 'r') as f:
    berlin_json = json.load(f)
berlin = gpd.GeoDataFrame([1], geometry=[shape(berlin_json['geometry'])], crs='EPSG:3035')

boundary = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')

# Load pop_grid

grid = gpd.read_parquet(PATH_PROCESSED / 'pop_grid.parquet')

# Build squares from points of grid
grid['geometry'] = grid.apply(lambda row: row.geometry.buffer(50, cap_style='square'), axis=1)
grid['index'] = grid.index

Repo root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/4. Semester/DEDA Project/DEDA_LLM_Spatial_Hotelling


In [2]:
# DB Station data
db_stations = pd.read_csv(PATH_RAW / 'db_station_data.csv')
db_stations = db_stations[db_stations['Bundesland'] == 'Berlin'].copy()
db_stations

,Bf-Nr,Aufgabenträger,Bahnhof,klasse,Bundesland,Anteil Serviceeinrichtung Stationspreis SPNV,Anteil Serviceeinrichtung Stationspreis SPFV
29,28,VBB Berlin,Ahrensfelde,4,Berlin,2.06,5.92
41,45,VBB Berlin,Albrechtshof,5,Berlin,2.02,5.77
51,53,VBB Berlin,Alexanderplatz,3,Berlin,3.93,11.32
101,116,VBB Berlin,Altglienicke,4,Berlin,2.06,5.92
108,7719,VBB Berlin,Alt-Reinickendorf,5,Berlin,2.02,5.77
...,...,...,...,...,...,...,...
5266,6824,VBB Berlin,Wittenau (Wilhelmsruher Damm),4,Berlin,2.06,5.92
5295,6871,VBB Berlin,Wollankstraße,4,Berlin,2.06,5.92
5312,6899,VBB Berlin,Wuhlheide,5,Berlin,2.02,5.77
5356,6967,VBB Berlin,Yorckstraße,5,Berlin,2.02,5.77


In [3]:
# The OSM stations query has been moved into osm.py as `_STATIONS_TAGS`.
# It fetches all elements tagged railway=station (S-Bahn, U-Bahn, regional,
# and long-distance rail) using `out geom tags;` — equivalent to the original
# query but with inline geometry, so no manual node-ref assembly is needed.
#
# Results are cached to data/raw/OSM_POIs_Berlin_stations.parquet.

from hotelling.spatial.osm import fetch_pois

osm_stations = fetch_pois(type="stations", city="Berlin")
print(f"OSM stations: {len(osm_stations)} rows, {osm_stations.shape[1]} columns")
print(f"Columns: {list(osm_stations.columns)}")
osm_stations.head()


OSM stations: 283 rows, 116 columns
Columns: ['osm_id', 'osm_type', 'geometry', 'contact:website', 'light_rail', 'name', 'network', 'network:short', 'network:wikidata', 'official_name', 'operator', 'public_transport', 'railway', 'railway:ref', 'railway:station_category', 'station', 'train', 'uic_ref', 'wheelchair', 'wikidata', 'wikipedia', 'contact:phone', 'departures_board', 'departures_board:speech_output', 'description', 'ref', 'addr:city', 'addr:housenumber', 'addr:postcode', 'addr:street', 'ref:ibnr', 'ref:station', 'uic_name', 'website', 'loc_name', 'note', 'start_date', 'toilets:wheelchair', 'subway', 'image', 'source', 'line', 'atm', 'atm:operator', 'operator:short', 'operator:wikidata', 'check_date:wheelchair', 'level', 'internet_access', 'surveillance', 'internet_access:fee', 'railway:ref:parent', 'source:address', 'phone', 'wikimedia_commons', 'surveillance:type', 'name:VBB', 'wheelchair:description', 'wheelchair:source', 'old_name', 'not:network:wikidata', 'url', 'layer', '

,osm_id,osm_type,geometry,contact:website,light_rail,name,network,network:short,network:wikidata,official_name,...,description:old_name,source:old_name,internet_access:ssid,shelter,name:ko,name:ru,railway:ref:BVG,ref_name,old_name1,point
0,21302157,node,POINT (13.33645 52.51437),http://www.s-bahn-berlin.de/fahrplanundnetz/ba...,yes,Tiergarten,Verkehrsverbund Berlin-Brandenburg,VBB,Q315451,Berlin-Tiergarten,...,None,None,None,None,None,None,None,None,None,POINT (13.33645 52.51437)
1,26124376,node,POINT (13.28442 52.51801),http://www.s-bahn-berlin.de/fahrplanundnetz/ba...,yes,Westend,Verkehrsverbund Berlin-Brandenburg,VBB,Q315451,Berlin-Westend,...,None,None,None,None,None,None,None,None,None,POINT (13.28442 52.51801)
2,26603219,node,POINT (13.17981 52.42141),None,yes,Wannsee,Verkehrsverbund Berlin-Brandenburg,VBB,Q315451,Berlin-Wannsee S-Bahn,...,None,None,None,None,None,None,None,None,None,POINT (13.17981 52.42141)
3,26943869,node,POINT (13.22763 52.51015),http://www.s-bahn-berlin.de/fahrplanundnetz/ba...,yes,Pichelsberg,Verkehrsverbund Berlin-Brandenburg,VBB,None,Berlin Pichelsberg,...,None,None,None,None,None,None,None,None,None,POINT (13.22763 52.51015)
4,27215654,node,POINT (13.26101 52.48825),http://www.s-bahn-berlin.de/fahrplanundnetz/ba...,yes,Grunewald,Verkehrsverbund Berlin-Brandenburg,VBB,None,Berlin-Grunewald S-Bahn,...,None,None,None,None,None,None,None,None,None,POINT (13.26101 52.48825)


In [ ]:
osm_stations

In [ ]:
from fuzzywuzzy import fuzz

# Create a list of station names from the OSM data
osm_station_names = osm_stations['official_name'].tolist()

# Create a list of station names from the DB data
db_stations
